# Project: monitoring with time series in pandas

Check two weeks of data-logger measurements: gaps, glitches, resampling and a report.

The full explanations are in [timeseries_pandas.md](timeseries_pandas.md). *This work is licensed under [Creative Commons Attribution 4.0 International License](http://creativecommons.org/licenses/by/4.0/).*

## How to use this notebook

* Run the cells **in order**, with `Shift+Enter`.
* Cells starting with `%%writefile` **create a file** (module, tests or script)
  in the folder of this notebook: like in the lesson, the project is made of
  real `.py` files, that you can also open in a text editor or Spyder.
* `%run script.py` runs a script as from the shell, and the cell with
  `pytest` runs the tests.
* Start JupyterLab from an **empty folder** dedicated to this project, in the
  `sci` environment:

  ```
  (sci) $ mkdir my_project
  (sci) $ cd my_project
  (sci) $ jupyter lab
  ```

## Some data

A temperature and a humidity every 10 minutes for two weeks, with 6 glitches and a 6-hour gap.

In [ ]:
%%writefile generate_logger.py
"""Generate two weeks of fake measurements from a data logger in a lab room."""

import numpy as np
import pandas as pd

rng = np.random.default_rng(seed=21)
time = pd.date_range("2026-03-02", "2026-03-16", freq="10min", inclusive="left")
hours = (time.hour + time.minute / 60).to_numpy()
day = np.arange(len(time)) / 144                       # days since the start

temperature = (21 + 2 * np.sin(2 * np.pi * (hours - 9) / 24)  # daily cycle
               + 0.15 * day                                    # slow drift
               + rng.normal(0, 0.2, len(time)))
humidity = 45 - 5 * np.sin(2 * np.pi * (hours - 9) / 24) + rng.normal(0, 1, len(time))

data = pd.DataFrame({"timestamp": time,
                     "temperature": temperature.round(2),
                     "humidity": humidity.round(1)})
spikes = rng.choice(len(data), 6, replace=False)       # electrical glitches
data.loc[spikes, "temperature"] = 85.0
data = data.drop(index=range(1000, 1036))              # logger offline for 6 h
data.to_csv("logger.csv", index=False)
print("Written logger.csv with", len(data), "rows")

In [ ]:
%run generate_logger.py

## Step 1: dates as index

`parse_dates` converts the text into real dates, and `index_col` uses them as
the index: rows can then be selected by date.

In [ ]:
%%writefile timeseries.py
"""Tools to check and summarize time series from a data logger."""

import pandas as pd


def load(filename):
    """Read a logger CSV file, with the timestamps as index (sorted)."""
    data = pd.read_csv(filename, parse_dates=["timestamp"], index_col="timestamp")
    return data.sort_index()


def find_gaps(index, expected="10min"):
    """Return the gaps of a time index larger than the expected interval.

    Returns
    -------
    pandas.DataFrame
        One row per gap: the last time before the gap, the first time after
        it, and the duration between them.
    """
    steps = index.to_series().diff()
    gaps = steps[steps > pd.Timedelta(expected)]
    return pd.DataFrame({"start": gaps.index - gaps,
                         "end": gaps.index,
                         "duration": gaps}).reset_index(drop=True)


def remove_spikes(series, window=7, threshold=3.0):
    """Replace isolated outliers by NaN.

    A value is an outlier when it differs from the rolling median of its
    neighbours by more than `threshold` (in the units of the series).
    """
    median = series.rolling(window, center=True, min_periods=1).median()
    return series.where((series - median).abs() <= threshold)


def daily_summary(series):
    """Return the minimum, mean and maximum of each day."""
    return series.resample("D").agg(["min", "mean", "max"]).round(2)


def time_above(series, limit, step="10min"):
    """Return the total time during which the series is above a limit."""
    return (series > limit).sum() * pd.Timedelta(step)

In [ ]:
from timeseries import load, find_gaps, remove_spikes, daily_summary, time_above

data = load("logger.csv")
data.head(3)

In [ ]:
data.loc["2026-03-05 12:00":"2026-03-05 12:30"]

In [ ]:
data["temperature"].max()

In [ ]:
find_gaps(data.index)

In [ ]:
clean = remove_spikes(data["temperature"])
print(clean.isna().sum(), "spikes removed; new maximum:", clean.max())
daily_summary(clean).head(3)

In [ ]:
time_above(clean, 23)

## Step 2: tests

In [ ]:
%%writefile test_timeseries.py
"""Tests for timeseries.py."""

import numpy as np
import pandas as pd

from timeseries import daily_summary, find_gaps, remove_spikes, time_above


def regular_index(n, start="2026-01-01"):
    return pd.date_range(start, periods=n, freq="10min")


def test_no_gap_in_regular_index():
    assert len(find_gaps(regular_index(50))) == 0


def test_find_one_gap():
    index = regular_index(50).delete(range(10, 16))      # remove 1 hour
    gaps = find_gaps(index)
    assert len(gaps) == 1
    assert gaps.loc[0, "duration"] == pd.Timedelta("70min")
    assert gaps.loc[0, "start"] == index[9]


def test_remove_spikes():
    series = pd.Series([20.0] * 10, index=regular_index(10))
    series.iloc[4] = 85.0
    clean = remove_spikes(series)
    assert np.isnan(clean.iloc[4])
    assert clean.drop(clean.index[4]).eq(20.0).all()


def test_daily_summary():
    index = pd.date_range("2026-01-01", periods=48, freq="1h")    # two days
    series = pd.Series(np.r_[np.full(24, 10.0), np.full(24, 20.0)], index=index)
    daily = daily_summary(series)
    assert daily["mean"].tolist() == [10.0, 20.0]


def test_time_above():
    series = pd.Series([1, 5, 6, 2], index=regular_index(4))
    assert time_above(series, 4) == pd.Timedelta("20min")

In [ ]:
import sys
!{sys.executable} -m pytest -q

## Step 3: the report

In [ ]:
%%writefile report_logger.py
"""Check and summarize the measurements of a data logger."""

import argparse

import matplotlib.pyplot as plt

from timeseries import daily_summary, find_gaps, load, remove_spikes, time_above


def main():
    parser = argparse.ArgumentParser(description="Data logger report.")
    parser.add_argument("filename", help="CSV file: timestamp, temperature, ...")
    parser.add_argument("--limit", type=float, default=23,
                        help="temperature limit (°C)")
    args = parser.parse_args()

    data = load(args.filename)
    print(f"{len(data)} measurements from {data.index[0]} to {data.index[-1]}")

    gaps = find_gaps(data.index)
    print(f"{len(gaps)} gap(s) in the recording:")
    for _, gap in gaps.iterrows():
        print(f"  no data from {gap['start']} to {gap['end']} ({gap['duration']})")

    temperature = remove_spikes(data["temperature"])
    n_spikes = temperature.isna().sum()
    print(f"{n_spikes} spike(s) removed")
    print("Time above the limit:", time_above(temperature, args.limit))

    daily = daily_summary(temperature)
    daily.to_csv("daily_temperature.csv")
    print("Daily summary saved in daily_temperature.csv")

    fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    axes[0].plot(data.index, data["temperature"], color="lightgray", label="raw")
    axes[0].plot(temperature.index, temperature, label="without spikes")
    axes[0].plot(temperature.rolling("6h").mean(), label="6 h rolling mean")
    axes[0].axhline(args.limit, color="red", linestyle="--", label="limit")
    for _, gap in gaps.iterrows():
        axes[0].axvspan(gap["start"], gap["end"], color="orange", alpha=0.3)
    axes[0].set_ylim(15, 30)
    axes[0].set_ylabel("Temperature (°C)")
    axes[0].legend(loc="upper left")
    axes[1].fill_between(daily.index, daily["min"], daily["max"], alpha=0.3,
                         step="post", label="daily min-max")
    axes[1].step(daily.index, daily["mean"], where="post", label="daily mean")
    axes[1].set_ylabel("Temperature (°C)")
    axes[1].legend(loc="upper left")
    fig.tight_layout()
    fig.savefig("logger_report.png", dpi=150)
    plt.show()


if __name__ == "__main__":
    main()

In [ ]:
%run report_logger.py logger.csv

**Look at the figure**: matplotlib draws a straight line across the gap,
and the daily mean of March 8 is too high, because the missing hours are
night hours. Missing data can hide in figures and bias statistics.

## Going further

**1.** Compute the correlation between temperature and humidity (after removing the spikes).

<details><summary>Answer</summary>

```python
data["temperature"] = remove_spikes(data["temperature"])
data.corr().round(2)
```

</details>

In [ ]:
# Your code here

**2.** Count the **episodes** above 23 °C, on the cleaned series and on its 1-hour rolling mean.

<details><summary>Answer</summary>

```python
def count_episodes(series, limit):
    above = series > limit
    starts = above & ~above.shift(1, fill_value=False)
    return starts.sum()

print(count_episodes(clean, 23), count_episodes(clean.rolling("1h").mean(), 23))
```

48 versus 15: noise makes the series cross the limit back and forth.

</details>

In [ ]:
# Your code here

**3.** Make the series regular with `resample("10min").mean()`, then fill only the short holes with `interpolate(limit=3)`. How many values remain missing?

<details><summary>Answer</summary>

```python
regular = clean.resample("10min").mean()
print(regular.isna().sum(), regular.interpolate(limit=3).isna().sum())
```

</details>

In [ ]:
# Your code here